# Module 5 — Inverse Kinematics
## Unit 1 — The Inverse Problem
### Lesson 1.1 — From Forward to Inverse

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
Forward = evaluate; inverse = solve. One-joint arm: position (L cosθ, L sinθ).


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(theta1, theta2, L1, L2):
    """Forward kinematics: planar 2-link gripper position."""
    x = L1*np.cos(theta1) + L2*np.cos(theta1+theta2)
    y = L1*np.sin(theta1) + L2*np.sin(theta1+theta2)
    return np.array([x, y])

def reachable(x, y, L1, L2, tol=1e-9):
    r = np.hypot(x, y)
    return abs(L1-L2)-tol <= r <= L1+L2+tol

def ik_two_link(x, y, L1, L2):
    """Return both (theta1,theta2) solutions; [] if unreachable, one if on boundary."""
    r2 = x*x + y*y
    c2 = (r2 - L1*L1 - L2*L2)/(2*L1*L2)
    if c2 < -1-1e-9 or c2 > 1+1e-9:
        return []
    c2 = max(-1.0, min(1.0, c2))
    sols=[]
    for sign in (+1.0, -1.0):
        t2 = sign*np.arccos(c2)
        t1 = np.arctan2(y, x) - np.arctan2(L2*np.sin(t2), L1+L2*np.cos(t2))
        sols.append((t1, t2))
        if abs(np.sin(t2)) < 1e-6:    # boundary: the two coincide
            break
    return sols

L=0.5
# Forward: plug in theta
theta=np.radians(30)
print("forward: theta=30deg ->", np.round([L*np.cos(theta), L*np.sin(theta)],3))

# Inverse (one joint): recover theta from a target on the radius-L circle
def ik_one_joint(x, y, L, tol=1e-9):
    if abs(np.hypot(x,y)-L) > tol: return None
    return np.arctan2(y, x)

t = ik_one_joint(0.433, 0.250, L, tol=1e-3)
print("inverse: target (0.433,0.250) -> theta(deg)=", round(np.degrees(t),2))


## Coding Exercise (§8)
Confirm fk(ik(target)) reproduces a reachable target.


In [ ]:
# YOUR CODE HERE
def fk_one(theta, L): return np.array([L*np.cos(theta), L*np.sin(theta)])
target=np.array([0.0, 0.5])
theta=np.arctan2(target[1], target[0])
assert np.allclose(fk_one(theta, L), target)
print("round-trip OK: theta(deg)=", round(np.degrees(theta),2))


## Check your work


In [ ]:
assert round(np.degrees(np.arctan2(0.250,0.433)),0)==30
assert np.allclose([L*np.cos(np.radians(30)), L*np.sin(np.radians(30))],[0.433,0.25],atol=1e-3)
print("All checks passed.")
